# 01 - The frame index

**Purpose.** An index of the ~15,000 historic frames on `Z:` — material of *varied
reliability*, shot across a year of ordinary imaging before this project or any of its
conventions existed. It exists so those frames can serve as **test data**: real pixels to
exercise the library against, and the route into the NGC 7000 exposure-ladder set.

**What it is not.** Not a measurement of the sensor, and not a dataset that owes this project
anything. Mixed setpoints, mixed gains and labels that disagree with the pixels are properties
of the corpus, not faults to correct. Frames captured *for* this project are shot to a protocol
and land in `data/` (`DECISIONS` D36).

**One job: build `results/frame_index.csv`.**

The archive holds ~15,000 frames on `Z:`, and every question this project asks starts with
"which frames?" - which darks match this light, which gains were actually used, where the ladder
has gaps. Answering that by walking the filesystem each time costs minutes of SMB latency per
question, so it is answered once into a CSV and read from there forever after.

The index is **one row per frame**, and it is built by *reading the pixels*, not by trusting the
folder or the `IMAGETYP` header. That is D18, and it is not paranoia: dark folders in this
archive mix gain and temperature inside a single exposure folder, and a number of flats and
darks were captured under a `Light` subframe type. Capture settings in the header - gain, offset,
exposure, set-temp, achieved temp - are trusted. The type label is evidence, not truth.

**The division of labour** (D35): `astropix` describes *one frame*. The loop over fifteen
thousand of them - what to skip, when to checkpoint, what to print - is orchestration, and it
lives here where you can read it and change it.

| output | written by |
|---|---|
| `results/frame_index.csv` | the scan cell below, then the classification cell after it |

The scan runs in **two passes**: read every frame, then measure the pedestal from the bias
frames that pass found and classify against it (D50). The second pass opens nothing.

**Cost: roughly 100 minutes** over SMB on a first pass, dominated by opening each frame. It is
incremental and resumable: interrupting it is safe, and re-running reads only what changed.

*This notebook imports `astropix` once. Restart the kernel after any change to the
package.*

In [ ]:
import os, pathlib, sys, time
import datetime as dt

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd

from astropix import fits as F, stats as ST

pd.set_option("display.width", 200)

RESULTS = pathlib.Path("..") / "results"
INDEX = RESULTS / "frame_index.csv"

# The four type folders, not their parent: `_by_type` also holds `_canon`,
# frames from a retired camera, with its own bias/dark/flat/light beneath it.
# Naming the four is the cheap guard; `scan_frame` checking INSTRUME on every
# frame is the one that cannot be defeated by a folder rename (D26).
ARCHIVE = pathlib.Path(r"Z:\pix\_astro\raw\_by_type")
ARCHIVE_ROOTS = [ARCHIVE / t for t in ("bias", "dark", "flat", "light")]

PROGRESS = 100      # frames between progress lines
CHECKPOINT = 200    # frames *read* between saves

RESULTS.mkdir(exist_ok=True)
for r in ARCHIVE_ROOTS:
    print(f"{'ok     ' if r.is_dir() else 'MISSING'}  {r}")

## What the library does, on one frame

`F.scan_frame` is the whole of the per-frame work: sample a few row-blocks, reduce them to
features per CFA sub-plane (D4), classify from those features, and check the frame came from
this rig. Everything below is a loop around this one call.

**Every pixel number in this index is in ADC counts** - `stats.frame_features` applies the one
conversion this project has, once, after checking the container is what we think it is
(`CLAUDE.md`, D41). Full scale is 4095, the gain-252 pedestal is 77, and header `EGAIN` applies
directly with no factor to remember.

Three groups of numbers come back, and they are not equally load-bearing:

- **`level`, and only `level`.** It is the single feature `classify` reads (D50): how far the
  frame sits above the pedestal for its gain is what "how much light arrived" means. Everything
  else below is description.
- **description** - `sigma`, `block_spread`, `mult16_frac`, `sat_frac`, and the per-plane
  medians `med_*` for R, G1, G2, B. Measured and stored because they make the corpus queryable
  and because two of them are integrity checks, but read by nothing in `classify`. The medians
  are the index's only colour, and a calibration frame whose planes disagree has a light leak.
  The per-plane *spreads* are computed - D4 is about how a statistic is computed - but only
  their mean is stored, as `sigma`: across 2,497 zero-light frames the four are identical to the
  digit in every one, so four columns record what one does.
- **whole-frame summary** - `mean`, `median`, `min`, `max`, `std`, `sampled_px`. Context for
  orientation and for notebook `02`. `std` here is pooled *across* the CFA planes, so on
  anything with colour in it it measures channel balance rather than noise - a median 14.5x
  `sigma` on flats, against 0.6-1.2x on bias and dark, which have no colour. `02` shows it next
  to `sigma`.

`sampled_px` is the honest caveat on all of it: frames are sampled, not read whole, so none of
these will match a full-frame tool like PixInsight, and they are not meant to.

**One frame cannot classify itself.** `classify` needs the pedestal for the frame's gain, and
the pedestal is measured from bias frames - many of them, which is a loop, which is this
notebook's job and not the library's (D35). So the scan below runs in **two passes**: read every
frame, then measure the pedestal from what was read and classify. Called with no pedestal, as
here, `scan_frame` says `unknown` rather than guessing.

In [ ]:
demo = next(p for p in ARCHIVE_ROOTS[1].rglob("*")
            if p.suffix.lower() in F.FITS_SUFFIXES)
rec = F.scan_frame(demo)          # no pedestal: one frame does not have one

print(demo.name, "\n")
for k in ("gain", "exptime", "ccd_temp", "imagetyp", "instrume"):
    print(f"  {k:12s} {rec[k]!r}")
print("\n  -- the one feature `classify` reads (ADC counts) --")
print(f"  {'level':12s} {rec['level']:.6g}")
print("\n  -- description and integrity: measured, stored, read by no branch --")
for k in ("sigma", "block_spread", "mult16_frac", "sat_frac"):
    print(f"  {k:12s} {rec[k]:.6g}")
print("\n  -- whole-frame summary, pooled across planes --")
for k in ("mean", "median", "min", "max", "std", "sampled_px"):
    print(f"  {k:12s} {rec[k]:.6g}")
print("\n  -- per-plane medians, R G1 G2 B (the index's only colour) --")
print("  med          " + " ".join(f"{rec['med_' + p]:8.1f}" for p in ("r", "g1", "g2", "b")))
print()
for k in ("measured_type", "declared_type", "type_agrees", "status"):
    print(f"  {k:12s} {rec[k]!r}")

## Finding the frames

Walking is cheap; reading is not. This pass does no `stat` and opens nothing - it exists to get
a denominator, so the progress report below can say "of how many".

In [ ]:
started = time.monotonic()
paths = sorted(str(p) for root in ARCHIVE_ROOTS for p in root.rglob("*")
               if p.suffix.lower() in F.FITS_SUFFIXES)
print(f"{len(paths)} frames found in {time.monotonic() - started:.1f} s")

previous = {}
if INDEX.exists():
    # dtype=str throughout: `mtime` is compared for exact equality against a
    # repr, and letting pandas infer a float here is precisely the round-trip
    # that would silently make every refresh a full re-read.
    previous = (pd.read_csv(INDEX, dtype=str)
                .set_index("path", drop=False).to_dict("index"))
print(f"{len(previous)} already indexed")

## The scan

Three rules worth reading before it runs, all of them D19:

- **A frame is opened only if it changed.** `F.needs_rescan` compares stored size and mtime
  against the file, and re-reads anything whose last pass did not end `ok`.
- **Rows are never deleted.** A path that has vanished is marked `missing`, so a number
  published months from now stays traceable to the frame it was measured on.
- **The write is atomic.** A temp file plus `os.replace`, so a reader always sees either the
  previous complete index or the new one, never half of one.

**Before running:** the archive must be frozen. New frames belong in `raw\_inbox\`, not in the
archive, until the run finishes.

**When the library changes, delete the index first.** The incremental rule above keys on *file
identity* - size and mtime - so it answers "did the frame change?" and never "did the code
change?". Nothing on `Z:` moves when `frame_features` gains a column or changes a unit, so a
re-run would skip all 15,000 rows and report a cheerful `0 read this pass` over a stale schema.
Deleting `results/frame_index.csv` is what forces the full pass; the last check in this notebook
is there to catch it if you forget.

This pass leaves `measured_type` as `unknown` on every row. That is not a failure - it is the
first of two passes, and the next cell is the second.

In [ ]:
HEAD = ["path", "size", "mtime", "indexed_at", "status",
        "measured_type", "declared_type", "type_agrees"]


def write_index(rows):
    """Atomic: a reader sees the old file or the new one, never half of one."""
    df = pd.DataFrame(list(rows.values()))
    df = df[[c for c in HEAD if c in df.columns]
            + [c for c in df.columns if c not in HEAD]]
    tmp = INDEX.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, INDEX)


def report(done, total, scanned, failed):
    elapsed = time.monotonic() - started
    rate = done / elapsed if elapsed else 0
    eta = (total - done) / rate if rate else 0
    print(f"  {done:>6}/{total} ({done / max(total, 1):5.1%})  "
          f"read {scanned}, skipped {done - scanned}, {failed} unreadable"
          f"  | {elapsed / 60:5.1f} min elapsed, ~{eta / 60:.0f} min left",
          flush=True)


rows = dict(previous)
stamp = dt.datetime.now().isoformat(timespec="seconds")
scanned = failed = 0
started = time.monotonic()

for done, path in enumerate(paths, 1):
    if F.needs_rescan(path, previous.get(path)):
        row = dict(F.stat_row(path), indexed_at=stamp, status="ok")
        try:
            row.update(F.scan_frame(path))
        except Exception as exc:          # truncated, zero-byte, unreadable
            row["status"] = "unreadable: " + type(exc).__name__
            failed += 1
        rows[path] = row
        scanned += 1
        if scanned % CHECKPOINT == 0:
            write_index(rows)
    if done % PROGRESS == 0:
        report(done, len(paths), scanned, failed)

# Never dropped, only marked -- a published constant must stay traceable to the
# frame it was measured on, even after someone reorganises the archive (D19).
here = set(paths)
gone = [p for p, r in rows.items() if p not in here and r.get("status") == "ok"]
for p in gone:
    rows[p].update(status="missing", indexed_at=stamp)

write_index(rows)
report(len(paths), len(paths), scanned, failed)
print(f"\nindex: {len(rows)} rows, {scanned} read this pass, {failed} unreadable, "
      f"{len(gone)} newly missing -> {INDEX}")

## The pedestal, and the types

The second pass costs no I/O: every number it needs is already in the rows above.

**Why a second pass at all.** `classify` asks how far a frame sits above the pedestal for its
gain - the level the camera reads with no light and no time. That is a per-gain constant, and
measuring it means looking at every bias frame, which is a loop over frames and therefore lives
here (D35). The library classifies one frame given the pedestal; it does not go and find one.

**Why this is not circular.** Bias frames are selected by exposure time alone
(`exptime <= ST.BIAS_MAX_EXPTIME`), which is a *trusted capture setting* - D18 distrusts the
type label, never the settings the camera was actually given. No pixel argument is made to
decide what a bias is, so the pedestal that comes out owes nothing to the classifier that
consumes it.

**Why the pedestal is not a constant in the library.** Two reasons, and the second is the one
that matters. It is not a number this project has measured to publication standard yet - that is
the offset sweep, build step 3, and hard-coding 65 and 77 here would put an unprovenanced
constant inside the code that produced the index. And a gain this archive happens not to contain
would break it: passing the pedestal in means the same three thresholds work at every gain, and
a gain with no bias frames behind it gets `unknown` instead of a guess.

In [ ]:
scan = pd.DataFrame(list(rows.values()))
for c in ("exptime", "gain", "level"):
    scan[c] = pd.to_numeric(scan[c], errors="coerce")

readable = scan[scan.status == "ok"]
bias = readable[readable.exptime <= ST.BIAS_MAX_EXPTIME]
pedestal = bias.groupby("gain")["level"].median()

print(f"{len(bias)} bias frames (exptime <= {ST.BIAS_MAX_EXPTIME} s), "
      f"pedestal per gain in ADC counts:")
print(bias.groupby("gain")["level"]
      .agg(n="size", min="min", median="median", max="max", std="std").to_string())

absent = sorted(set(readable.gain.dropna()) - set(pedestal.index))
if absent:
    print(f"\nno bias frames at gain {absent} -- those frames stay 'unknown', "
          "which is the honest answer and not a bug")

for r in readable.itertuples():
    row = rows[r.path]
    ped = None if pd.isna(r.gain) else pedestal.get(r.gain)
    row["measured_type"] = ST.classify({"level": r.level}, r.exptime, ped)
    declared = row.get("declared_type") or ""
    row["type_agrees"] = (declared == row["measured_type"]) if declared else None

write_index(rows)
print(f"\nclassified {len(readable)} readable rows -> {INDEX}")

## Did it work?

Three questions, in the order in which a wrong answer would hurt most: did everything get read,
does the classifier agree with the labels, and is the bit-shift what we think it is.

In [ ]:
idx = pd.read_csv(INDEX)
ok = idx[idx.status == "ok"]

print(f"{len(idx)} rows, {len(ok)} readable, "
      f"indexed {idx.indexed_at.min()} .. {idx.indexed_at.max()}\n")
print("status:")
print(idx.status.str.split(":").str[0].value_counts().to_string())
print("\nmeasured type vs declared label:")
print(pd.crosstab(ok.declared_type, ok.measured_type, margins=True).to_string())

A disagreement here is not a bug - it is the archive telling you a frame is mislabelled,
and that list is the seed of the archive cleanup (D20). What *would* be a bug is a clean
diagonal with nothing off it: that would mean the classifier is reading the label it was built
to ignore.

In [ ]:
print("fraction of pixel values that are exact multiples of 16:")
print(ok.mult16_frac.agg(["min", "mean", "max"]).to_string())

# The guard against a stale schema surviving the incremental rule: a row still
# carrying stored values would top out near 65535, not 4095.
ceiling = ok["max"].max()
print(f"\nhighest sampled pixel anywhere: {ceiling:.0f} counts "
      f"(full scale {4095})")
assert ceiling <= 4095, "a stored-unit row survived -- delete the index and rescan"

print("\ngain vs measured type:")
print(pd.crosstab(ok.gain, ok.measured_type, margins=True).to_string())

`mult16_frac = 1.0` everywhere is the evidence that licenses the unit this whole project works
in. The camera digitises to 12 bits and stores the value shifted left by four, so the fifteen
values between each stored multiple of 16 are unreachable - a stored file value is 16x the
number the ADC actually produced. Measured, not assumed, and measured on the *stored* values
before the conversion, because reading it off converted data would be circular.

Because that check passes, everything in this index is divided down once and published in **ADC
counts** (D41): full scale 4095, the gain-252 pedestal 77, and header `EGAIN` - quoted per ADC
count - applying directly with no factor to carry. The assertion above is the standing guard on
it: a row that somehow arrived in stored units would show a maximum near 65520 and stop the
notebook rather than quietly sit in the CSV being 16x wrong.

**What changed in this build (D50).** The classifier used to decide dark from light by asking
whether a frame's bright pixels were *shaped* like stars, at a threshold of five times the
frame's own MAD. That threshold rose with the sky, so the brighter the sky the more star-blind
the test became, and 484 real lights - including 141 shot through trees and cloud, which have no
stars at all - were published as `dark`. Level above the pedestal separates the same 12,642
frames with no overlap and no reference to stars, so `classify` now reads one number, `light` is
the fallback rather than `dark`, and `tail_frac`, `clump_frac`, `clump_h` and `clump_v` are gone
from the index. `spatial.bright_pixels` remains, unused by this notebook, for the hot-pixel work
it was always the right tool for.

---

**Next.** The index is a map, not a measurement - nothing here says what the sensor *does*. That
needs a real noise estimator and a photon transfer curve, and the order those arrive in is open
(`DECISIONS` D33).